# Breast Cancer Classification

## IDC Histopathology Image Classification

### Objective

Build a deep learning model to classify breast histopathology image patches into:

- Class 0 → Non-IDC / Benign
- Class 1 → IDC / Malignant

### Workflow

1. Import libraries
2. Explore dataset
3. Build image DataFrame
4. Perform patient-level train/validation/test split
5. Analyze class distribution
6. Create TensorFlow data pipeline
7. Apply image augmentation
8. Build CNN model
9. Train model
10. Evaluate model
11. Generate confusion matrix
12. Calculate classification metrics
13. Save trained model
14. Build Streamlit application

In [1]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.21.0
Num GPUs Available: 0


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "IDC_regular_ps50_idx5"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

In [ ]:
image_extensions = {".png", ".jpg", ".jpeg"}

image_files = []

for file_path in DATA_DIR.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() in image_extensions:
        image_files.append(file_path)

print("Total images:", len(image_files))

In [ ]:
records = []

for image_path in image_files:
    filename = image_path.name

    if "class0" in filename:
        label = 0
    elif "class1" in filename:
        label = 1
    else:
        continue

    # Patient ID is generally the first component
    patient_id = filename.split("_")[0]

    records.append({
        "image_path": str(image_path),
        "filename": filename,
        "patient_id": patient_id,
        "label": label
    })

df = pd.DataFrame(records)

print("Total usable images:", len(df))
df.head()

In [ ]:
print("Dataset shape:", df.shape)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nPatient count:")
print(df["patient_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df["class_name"] = df["label"].map({
    0: "Non-IDC",
    1: "IDC"
})

df.head()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="class_name"
)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")

plt.tight_layout()
plt.show()

In [ ]:
sample_df = df.sample(
    n=min(12, len(df)),
    random_state=SEED
)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):

    image = Image.open(row["image_path"])

    ax.imshow(image)

    ax.set_title(
        f'{row["class_name"]}\nPatient: {row["patient_id"]}'
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
patients = df["patient_id"].unique()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.30,
    random_state=SEED
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED
)

train_df = df[df["patient_id"].isin(train_patients)].copy()
val_df = df[df["patient_id"].isin(val_patients)].copy()
test_df = df[df["patient_id"].isin(test_patients)].copy()

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print()

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

In [ ]:
train_patient_set = set(train_df["patient_id"])
val_patient_set = set(val_df["patient_id"])
test_patient_set = set(test_df["patient_id"])

print(
    "Train ∩ Validation:",
    len(train_patient_set.intersection(val_patient_set))
)

print(
    "Train ∩ Test:",
    len(train_patient_set.intersection(test_patient_set))
)

print(
    "Validation ∩ Test:",
    len(val_patient_set.intersection(test_patient_set))
)

In [ ]:
print("TRAIN")
print(train_df["class_name"].value_counts(normalize=True))

print("\nVALIDATION")
print(val_df["class_name"].value_counts(normalize=True))

print("\nTEST")
print(test_df["class_name"].value_counts(normalize=True))

In [ ]:
IMG_SIZE = (96, 96)

BATCH_SIZE = 32

EPOCHS = 15

NUM_CLASSES = 2

AUTOTUNE = tf.data.AUTOTUNE

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    return image, label

In [ ]:
def create_dataset(dataframe, shuffle=False):

    paths = dataframe["image_path"].values
    labels = dataframe["label"].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 10000),
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:
train_ds = create_dataset(
    train_df,
    shuffle=True
)

val_ds = create_dataset(
    val_df
)

test_ds = create_dataset(
    test_df
)

print("TensorFlow datasets created.")

In [ ]:
images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation"
)

In [ ]:
sample_images, sample_labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for i in range(9):

    augmented_image = data_augmentation(
        sample_images[i:i+1],
        training=True
    )

    plt.subplot(3, 3, i + 1)

    plt.imshow(
        augmented_image[0].numpy()
    )

    plt.title(
        "IDC" if sample_labels[i] == 1 else "Non-IDC"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model = keras.Sequential(
    [
        layers.Input(
            shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        ),

        data_augmentation,

        layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(
            128,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(
            256,
            (3, 3),
            activation="relu",
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.4),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.Dropout(0.3),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ],
    name="breast_cancer_cnn"
)

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

print("Model compiled successfully.")

In [ ]:
model_path = MODELS_DIR / "breast_cancer_cnn.keras"

callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks ready.")

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
history_df = pd.DataFrame(history.history)

history_df.head()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend()

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "accuracy_curve.png",
    dpi=150
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "loss_curve.png",
    dpi=150
)

plt.show()

In [ ]:
best_model = keras.models.load_model(
    model_path
)

print("Best model loaded.")

In [ ]:
test_results = best_model.evaluate(
    test_ds,
    return_dict=True
)

print("\nTest Results")

for metric, value in test_results.items():
    print(
        f"{metric}: {value:.4f}"
    )

In [ ]:
y_true = test_df["label"].values

y_probability = best_model.predict(
    test_ds,
    verbose=1
).ravel()

y_pred = (
    y_probability >= 0.5
).astype(int)

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Non-IDC",
            "IDC"
        ],
        digits=4
    )
)

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    y_probability
)

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],

    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]
})

metrics_df

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=[
        "Non-IDC",
        "IDC"
    ],
    yticklabels=[
        "Non-IDC",
        "IDC"
    ]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "confusion_matrix.png",
    dpi=150
)

plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_probability
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"ROC-AUC = {auc:.4f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "roc_curve.png",
    dpi=150
)

plt.show()

In [ ]:
metrics_df.to_csv(
    RESULTS_DIR / "metrics.csv",
    index=False
)

print("Metrics saved.")

In [ ]:
def predict_image(
    image_path,
    model,
    image_size=IMG_SIZE
):

    image = Image.open(image_path).convert("RGB")

    image = image.resize(image_size)

    image_array = np.array(image).astype(
        np.float32
    ) / 255.0

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    probability = model.predict(
        image_array,
        verbose=0
    )[0][0]

    if probability >= 0.5:
        prediction = "IDC"
    else:
        prediction = "Non-IDC"

    return prediction, float(probability)

In [ ]:
test_image_path = test_df.iloc[0]["image_path"]

prediction, probability = predict_image(
    test_image_path,
    best_model
)

print("Image:", test_image_path)
print("Prediction:", prediction)
print("Probability:", probability)

In [ ]:
image = Image.open(test_image_path)

plt.figure(figsize=(6, 6))

plt.imshow(image)

plt.title(
    f"Prediction: {prediction}\n"
    f"Probability: {probability:.4f}"
)

plt.axis("off")

plt.show()

In [ ]:
final_model_path = (
    MODELS_DIR / "breast_cancer_classifier.keras"
)

best_model.save(
    final_model_path
)

print(
    "Model saved at:",
    final_model_path
)